<a href="https://colab.research.google.com/github/Hatoo22/2026-GP1-14/blob/main/modernbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification using **ModernBERT**

In this project, we build a model to classify text comments into multiple toxicity categories (multi-label classification), such as:
- Bullying
- Threats
- Hate speech
- Sexual harassment

- other toxicity

We use Natural Language Processing (NLP) techniques to process text and train a deep learning model to detect toxic patterns.

###**Model: ModernBERT**

In this project, we used the **ModernBERT-large** model, which is an advanced version of BERT.  
It is designed to better understand language context, making it highly effective for NLP tasks like toxicity classification.

It is based on the Transformer architecture and provides deep contextual representations of text, which helps it:
- Understand the full meaning of sentences
- Distinguish between different types of toxicity
- Handle complex and long text inputs

# **Install Required Libraries**


In this step, we install and update the required libraries for the project.  
These libraries include tools for data processing, machine learning, and AI models such as transformers.  
If the libraries are already installed, you will see "already satisfied".

In [ ]:
!pip install -U "transformers>=4.48.0" datasets scikit-learn openpyxl accelerate

#Import Libraries and Setup


This section imports all required libraries for building a multi-label classification model using deep learning.  
It includes tools for data processing, dataset splitting, evaluation metrics, and pretrained models from Hugging Face.

We also define key configurations such as:
- Model name
- Data file path
- Column names (text and labels)
- Training parameters like batch size and number of epochs

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

warnings.filterwarnings("ignore")

# ========= settings =========
MODEL_NAME = "answerdotai/ModernBERT-large"
FILE_PATH = "final_data_for_processing.xlsx"
TEXT_COL = "comment_text"
LABEL_COLS = [
    "is_toxic",
    "threat",
    "bullying",
    "sexual_harassment",
    "hate_speech",
    "other_toxicity"
]

OUTPUT_DIR = "./modernbert_multilabel_output"
MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 2      # ModernBERT-large
EVAL_BATCH_SIZE = 2
GRAD_ACCUM = 8            # batch size
MAX_LENGTH = 512
LEARNING_RATE = 2e-5      #Learning rate
NUM_EPOCHS = 3          # Number of training epochs
WEIGHT_DECAY = 0.01
SEED = 42
THRESHOLD = 0.5           # threshold

# Set Random Seed

This section sets a fixed random seed to ensure reproducibility of results.  
This is very important in machine learning experiments so that results remain consistent across runs.

The seed is applied to:
- random
- numpy
- torch (GPU)


The data is shuffled randomly before training to prevent the model from learning any order-based patterns.
This is considered a **preprocessing step** because it prepares the data for better training.

In [ ]:
# Function to set random seed across all libraries

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# Load and Explore Data

In [ ]:
df = pd.read_excel(FILE_PATH)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (11222, 7)
Columns: ['comment', 'is_toxic', 'threat', 'bullying', 'sexual_harassment', 'hate_speech', 'other_toxicity']


,comment,is_toxic,threat,bullying,sexual_harassment,hate_speech,other_toxicity
0,"Massively underrated, massively difficult, mas...",0,0,0,0,0,0
1,This game is not so bad as people say. Of cour...,0,0,0,0,0,0
2,"Honestly, PUBG gets ugly fast. In ranked, solo...",1,0,1,0,0,1
3,"Great solo exploration game, I play this game ...",0,0,0,0,0,0
4,boring as hell you get about mins gameplay the...,0,0,0,0,0,0


# Split Dataset


This section splits the dataset into three parts:
- Training set
- Validation set
- Test set

First, the data is split into:
- 80% training
- 20% temporary (validation + test)

Then, the temporary set is split into:
- 10% validation
- 10% test

This helps to:
- Train the model effectively
- Evaluate performance during training
- Test the model on unseen data

In [ ]:
# Split data into training (80%) and temporary set (20%)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)
# Split temporary set into validation (10%) and test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    shuffle=True
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (8977, 7)
Validation: (1122, 7)
Test: (1123, 7)


**We prepare the data** for use with Hugging Face by selecting the text column and label columns,  
then converting the pandas DataFrame into a Dataset format.

This step is important because it:
- Makes the data compatible with the Trainer
- Simplifies tokenization and further processing

In [ ]:
TEXT_COL = "comment"
LABEL_COLS = [
    "is_toxic",
    "threat",
    "bullying",
    "sexual_harassment",
    "hate_speech",
    "other_toxicity"
]

def make_hf_dataset(dataframe):
    return Dataset.from_pandas(
        dataframe[[TEXT_COL] + LABEL_COLS].reset_index(drop=True)
    )

train_ds = make_hf_dataset(train_df)
val_ds = make_hf_dataset(val_df)
test_ds = make_hf_dataset(test_df)

# Tokenizer / Load Tokenizer

This section loads the tokenizer for the selected pretrained model.  


We use the same model name to ensure compatibility between:
- tokenizer
- model

The parameter use_fast=True enables the fast version for better performance.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# Tokens / Tokenization and Encoding


This section converts text data into numerical tokens using the tokenizer.  
It also prepares the multi-label targets for each example.

The process includes:
- Converting all text inputs to strings to avoid errors
- Tokenizing the text
- Setting maximum sequence length
- Creating label arrays for each sample

Then, this process is applied to:
- Training data
- Validation data
- Test data


Text is converted into numerical tokens so the model can understand it.
This is a key **preprocessing step** because it transforms text into a format suitable for the model.


Long texts are truncated to fit within the model’s maximum input length.
This is a **preprocessing step** because it ensures the data fits the model’s limitations.

In [ ]:
# Function to tokenize text and prepare labels
def tokenize_function(examples):
    # Ensure all text inputs are strings
    texts_to_tokenize = [str(text) for text in examples[TEXT_COL]]
        # Tokenize the text
    tokenized = tokenizer(
        texts_to_tokenize,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )
    tokenized["labels"] = [
        [float(examples[col][i]) for col in LABEL_COLS]
        for i in range(len(examples[TEXT_COL]))
    ]
    return tokenized

train_tokenized = train_ds.map(tokenize_function, batched=True, remove_columns=train_ds.column_names)
val_tokenized = val_ds.map(tokenize_function, batched=True, remove_columns=val_ds.column_names)
test_tokenized = test_ds.map(tokenize_function, batched=True, remove_columns=test_ds.column_names)

Map:   0%|          | 0/8977 [00:00<?, ? examples/s]

Map:   0%|          | 0/1122 [00:00<?, ? examples/s]

Map:   0%|          | 0/1123 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# **Load Model**

This section loads a pretrained classification model from Hugging Face.  
We use ModernBERT-large, which is suitable for NLP tasks.

In [ ]:
# Load pretrained sequence classification model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_COLS),
    problem_type="multi_label_classification"
)

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Compute Evaluation Metrics

This section computes evaluation metrics for the model in a multi-label classification setting.


In [ ]:
# Function to compute evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Handle invalid numerical values
    logits = np.nan_to_num(logits, nan=0.0, posinf=20.0, neginf=-20.0)
    # Convert logits to probabilities using Sigmoid
    probs = 1.0 / (1.0 + np.exp(-logits))
        # Apply threshold to get binary predictions
    preds = (probs >= THRESHOLD).astype(int)
        # Convert labels to integers
    labels = labels.astype(int)

    # Compute evaluation metrics
    metrics = {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "micro_precision": precision_score(labels, preds, average="micro", zero_division=0),
        "micro_recall": recall_score(labels, preds, average="micro", zero_division=0),
        "subset_accuracy": accuracy_score(labels, preds),
    }
    return metrics

# Training Configuration

This section defines all training configurations using TrainingArguments.


In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,

    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    save_total_limit=2,
    # Precision settings
    report_to="none",
    fp16=False,
    bf16=False,
    seed=SEED,
    dataloader_num_workers=2
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


This section initializes the Trainer, which manages the full training and evaluation process.

We connect all components:
- The model
- Training and validation datasets
- Training configurations
- Metrics function

We also add Early Stopping to stop training if performance does not improve,  
which helps to:
- Reduce overfitting
- Save time and resources

In [ ]:
# Initialize Trainer
trainer = Trainer(
    model=model,      # Model
    args=training_args,        # Training arguments
    train_dataset=train_tokenized,       # Training dataset
    eval_dataset=val_tokenized,      # Validation dataset
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# **Start Training**

This section starts the model training process using the Trainer.

The model will:
- Learn from the training data
- Evaluate on validation data after each epoch
- Automatically save the best model

The training output includes:
- Loss values
- Training steps
- Performance details

In [ ]:
train_result = trainer.train()
train_result

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Weighted F1,Micro Precision,Micro Recall,Subset Accuracy
1,0.563314,0.050640,0.968790,0.964978,0.968472,0.995419,0.943548,0.937611
2,0.382186,0.049871,0.968849,0.965223,0.968532,0.993481,0.945409,0.936720
3,0.126738,0.058817,0.965909,0.960884,0.965611,0.983290,0.949132,0.931373


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1686, training_loss=0.6067062484836239, metrics={'train_runtime': 3201.8778, 'train_samples_per_second': 8.411, 'train_steps_per_second': 0.527, 'total_flos': 5884255954721844.0, 'train_loss': 0.6067062484836239, 'epoch': 3.0})

# Validate Model Performance

This section evaluates the model on the validation dataset separately.

We create a new Trainer for evaluation to:
- Generate predictions
- Compute metrics
- Analyze model performance

Finally, we print evaluation results such as:
- F1 Score
- Precision
- Recall
- Accuracy

In [ ]:
# Create a Trainer for evaluation only
from transformers import Trainer

val_trainer = Trainer(
    model=trainer.model,
    args=training_args,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    processing_class=tokenizer
)

val_output = val_trainer.predict(val_tokenized)   # Generate predictions and metrics

# Print validation results
print("Validation Results:")
print(val_output.metrics)

Validation Results:
{'test_loss': 0.04987103492021561, 'test_model_preparation_time': 0.0136, 'test_micro_f1': 0.9688493324856962, 'test_macro_f1': 0.9652229277278562, 'test_weighted_f1': 0.9685315777311259, 'test_micro_precision': 0.9934810951760105, 'test_micro_recall': 0.9454094292803971, 'test_subset_accuracy': 0.9367201426024956, 'test_runtime': 74.0942, 'test_samples_per_second': 15.143, 'test_steps_per_second': 7.571}


# Final Test Evaluation

This section performs the final evaluation on the test dataset,  
which was not used during training or validation.

This provides a realistic estimate of model performance.

We:
- Use the trained model
- Generate predictions on test data
- Compute and display evaluation metrics

In [ ]:
from transformers import Trainer
# Create a Trainer for test evaluation
test_trainer = Trainer(
    model=trainer.model,
    args=training_args,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    processing_class=tokenizer
)

test_output = test_trainer.predict(test_tokenized)   # Generate predictions on test set

# Print test results
print("Test Results:")
print(test_output.metrics)

Test Results:
{'test_loss': 0.05863962322473526, 'test_model_preparation_time': 0.0031, 'test_micro_f1': 0.9665841584158416, 'test_macro_f1': 0.9620314177540451, 'test_weighted_f1': 0.9663930727270951, 'test_micro_precision': 0.9923761118170267, 'test_micro_recall': 0.9420989143546441, 'test_subset_accuracy': 0.9305431878895815, 'test_runtime': 41.765, 'test_samples_per_second': 26.889, 'test_steps_per_second': 13.456}


## **Detailed Classification Report for Each Label**


In [ ]:
from sklearn.metrics import classification_report

pred_output = trainer.predict(test_tokenized)
logits = np.nan_to_num(pred_output.predictions, nan=0.0, posinf=20.0, neginf=-20.0)
labels = pred_output.label_ids.astype(int)

probs = 1.0 / (1.0 + np.exp(-logits))
preds = (probs >= THRESHOLD).astype(int)

for i, label_name in enumerate(LABEL_COLS):
    print(f"\n========== {label_name} ==========")
    print(classification_report(labels[:, i], preds[:, i], zero_division=0))


========== is_toxic ==========
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       498
           1       1.00      0.97      0.98       625

    accuracy                           0.98      1123
   macro avg       0.98      0.98      0.98      1123
weighted avg       0.98      0.98      0.98      1123


========== threat ==========
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       964
           1       1.00      0.92      0.96       159

    accuracy                           0.99      1123
   macro avg       0.99      0.96      0.98      1123
weighted avg       0.99      0.99      0.99      1123


========== bullying ==========
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       840
           1       0.99      0.93      0.96       283

    accuracy                           0.98      1123
   macro avg       0.99      0.97 

# Analyze Results and Create DataFrame

In this section, we convert model predictions into a structured DataFrame for easier analysis.

We:
- Generate predictions from the model
- Convert logits into probabilities, then into binary predictions
- Create a table containing:
  - Original text
  - True labels
  - Predicted labels

We also add:
- A column indicating if the prediction is completely correct (all_correct)
- A column counting the number of errors per sample (num_errors)

This helps us:
- Analyze model errors
- Identify difficult cases
- Prepare results for visualization or saving

In [ ]:
import numpy as np
import pandas as pd

# Get model predictions on test data
test_output = trainer.predict(test_tokenized)
# Clean invalid numerical values
logits = np.nan_to_num(test_output.predictions, nan=0.0, posinf=20.0, neginf=-20.0)
labels = test_output.label_ids.astype(int)

# Convert logits to probabilities
probs = 1.0 / (1.0 + np.exp(-logits))
# Convert probabilities to binary predictions
preds = (probs >= THRESHOLD).astype(int)

# conver to DataFrame
results_df = pd.DataFrame()

# texts
results_df["text"] = test_df[TEXT_COL].reset_index(drop=True)

# Add true labels
for i, col in enumerate(LABEL_COLS):
    results_df[f"true_{col}"] = labels[:, i]

# Add predicted labels
for i, col in enumerate(LABEL_COLS):
    results_df[f"pred_{col}"] = preds[:, i]

# Check if all labels are correctly predicted
results_df["all_correct"] = (labels == preds).all(axis=1)

# Count number of errors per sample
results_df["num_errors"] = (labels != preds).sum(axis=1)

results_df.head()

,text,true_is_toxic,true_threat,true_bullying,true_sexual_harassment,true_hate_speech,true_other_toxicity,pred_is_toxic,pred_threat,pred_bullying,pred_sexual_harassment,pred_hate_speech,pred_other_toxicity,all_correct,num_errors
0,"I really really want to like this game, and gi...",0,0,0,0,0,0,0,0,0,0,0,0,True,0
1,The player culture in FiveM keeps overshadowin...,1,0,1,0,0,0,1,0,1,0,0,0,True,0
2,Does your platformer have meat it in? Didn't t...,0,0,0,0,0,0,0,0,0,0,0,0,True,0
3,"Honestly, Paladins gets ugly fast. The culture...",1,0,1,0,0,1,1,0,1,0,0,1,True,0
4,I remember when this came out and everyone was...,0,0,0,0,0,0,0,0,0,0,0,0,True,0


Display Full Results

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

results_df

,text,true_is_toxic,true_threat,true_bullying,true_sexual_harassment,true_hate_speech,true_other_toxicity,pred_is_toxic,pred_threat,pred_bullying,pred_sexual_harassment,pred_hate_speech,pred_other_toxicity,all_correct,num_errors
0,"I really really want to like this game, and gi...",0,0,0,0,0,0,0,0,0,0,0,0,True,0
1,The player culture in FiveM keeps overshadowin...,1,0,1,0,0,0,1,0,1,0,0,0,True,0
2,Does your platformer have meat it in? Didn't t...,0,0,0,0,0,0,0,0,0,0,0,0,True,0
3,"Honestly, Paladins gets ugly fast. The culture...",1,0,1,0,0,1,1,0,1,0,0,1,True,0
4,I remember when this came out and everyone was...,0,0,0,0,0,0,0,0,0,0,0,0,True,0
5,"At this point, EA FC gets ugly fast. The cultu...",1,1,1,0,1,0,1,0,1,0,1,0,False,1
6,"on the weekend, i do not think Marvel Rivals i...",1,0,0,0,1,1,1,0,0,0,1,1,True,0
7,this game always puts you in a team with dumbass,0,0,0,0,0,0,0,0,0,0,0,0,True,0
8,as much as this game is a piece of history for...,0,0,0,0,0,0,0,0,0,0,0,0,True,0
9,The graphics migth be outdated but it's still ...,0,0,0,0,0,0,0,0,0,0,0,0,True,0


**Save Results to File**

In [ ]:
file_name = "test_results_full.csv"
results_df.to_csv(file_name, index=False)

print(f"Saved as {file_name}")

Saved as test_results_full.csv


**Analyze Data Distribution**

In [ ]:
import pandas as pd

df = pd.read_excel("final_data_for_processing.xlsx")

toxic_count = (df["is_toxic"] == 1).sum()
print("عدد التوكسيك:", toxic_count)
toxic_count = (df["is_toxic"] == 0).sum()
print("عدد  التوكسيك:", toxic_count)

عدد التوكسيك: 5987
عدد  التوكسيك: 5235


## Manual Test on Custom Sentences

This section performs a manual test using custom-written sentences instead of the original test dataset.


In [ ]:
# =========================
# MANUAL TEST
# =========================

import torch
import numpy as np

texts = [
    "Every match in this game turns into people insulting the weakest player until they leave.",
    "I stopped using voice chat because some players kept threatening to find me and hurt me.",
    "The community is awful, they keep mocking new players for every small mistake.",
    "Some random guy in the lobby kept making disgusting sexual comments at me the whole game.",
    "I have seen players use racist slurs in chat whenever they get angry.",
    "The matchmaking is bad and the servers lag all the time.",
    "People in ranked keep telling others to uninstall and saying they are useless.",
    "One player said he would track me down if I queued against him again.",
    "The fanbase gets really defensive, but I would not call it toxic all the time.",
    "Women get harassed so much in this game's voice chat that many of them avoid speaking."
]

# Expected true labels for each sentence
true_labels = [
    [1,0,1,0,0,0],  # insulting weakest player
    [1,1,0,0,0,0],  # direct threat
    [1,0,1,0,0,0],  # mocking new players
    [1,0,0,1,0,0],  # sexual harassment
    [1,0,0,0,1,0],  # hate speech
    [0,0,0,0,0,0],  # technical complaint only
    [1,0,1,0,0,0],  # bullying
    [1,1,0,0,0,0],  # threat
    [0,0,0,0,0,0],  # opinion, not clearly toxicity
    [1,0,0,1,0,0]   # harassment toward women + hate element
]


# Use the trained model
model = trainer.model
model.eval()

# Get the model device
device = model.device

inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

# Move inputs to the same device as the model
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Convert logits to probabilities using Sigmoid
probs = torch.sigmoid(logits).cpu().numpy()
preds = (probs >= THRESHOLD).astype(int)

correct = 0

# Display results for each sentence
for i in range(len(texts)):
    print("\n====================")
    print("TEXT:", texts[i])
    print("TRUE:", true_labels[i])
    print("PRED:", preds[i].tolist())
    print("SCORES:", [round(x, 3) for x in probs[i].tolist()])

    # Check whether all labels are predicted correctly
    if (preds[i] == true_labels[i]).all():
        print("✅ CORRECT")
        correct += 1
    else:
        print("❌ WRONG")

print("\n====================")
print(f"Correct: {correct}/{len(texts)}")
print(f"Accuracy: {correct/len(texts):.2f}")


TEXT: Every match in this game turns into people insulting the weakest player until they leave.
TRUE: [1, 0, 1, 0, 0, 0]
PRED: [1, 0, 1, 0, 0, 0]
SCORES: [0.999, 0.0, 1.0, 0.0, 0.001, 0.036]
✅ CORRECT

TEXT: I stopped using voice chat because some players kept threatening to find me and hurt me.
TRUE: [1, 1, 0, 0, 0, 0]
PRED: [1, 1, 0, 0, 0, 0]
SCORES: [0.999, 1.0, 0.028, 0.001, 0.003, 0.146]
✅ CORRECT

TEXT: The community is awful, they keep mocking new players for every small mistake.
TRUE: [1, 0, 1, 0, 0, 0]
PRED: [1, 0, 1, 0, 0, 0]
SCORES: [1.0, 0.002, 1.0, 0.002, 0.001, 0.03]
✅ CORRECT

TEXT: Some random guy in the lobby kept making disgusting sexual comments at me the whole game.
TRUE: [1, 0, 0, 1, 0, 0]
PRED: [1, 0, 0, 1, 0, 0]
SCORES: [0.973, 0.019, 0.002, 0.997, 0.001, 0.033]
✅ CORRECT

TEXT: I have seen players use racist slurs in chat whenever they get angry.
TRUE: [1, 0, 0, 0, 1, 0]
PRED: [1, 0, 0, 0, 1, 0]
SCORES: [1.0, 0.038, 0.001, 0.003, 1.0, 0.132]
✅ CORRECT

TEXT: Th

The model achieved strong performance in the manual test, correctly classifying 9 out of 10 sentences with an accuracy of 90%. This indicates its high capability in detecting explicit toxicity patterns such as threats, bullying, and hate speech with high confidence. The model also demonstrated good distinction between toxic and non-toxic content, as it correctly handled neutral technical complaints. However, a minor limitation appeared in understanding more complex linguistic context, particularly in sentences involving negation, where it misclassified one sentence as toxic despite it expressing the opposite. Overall, the model shows excellent performance in clear cases, with slight room for improvement in handling nuanced and context-dependent language.